## Type constraint Repair Analyzer

In [ ]:
import pandas as pd

# You can read the already calculated repairs file, or read the one generated on 't-box repairs analyzer' folder with the script 't-box_repairs_analyzer.py'
df_type_repairs = pd.read_csv("../../datasets/type_repairs.csv")

df_type_repairs

- The cell below counts different types of basic T-box repairs generated with the relational database:

In [ ]:
print(len(df_type_repairs[(df_type_repairs['C_deleted'] == True)] ))
print(len(df_type_repairs[(df_type_repairs['C_deprecated'] == True)] ))
print(len(df_type_repairs[(df_type_repairs['CQ_added_exception'] == True)] ))

- checking base statement deletions:

In [ ]:
df_type_repairs['S_deleted'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def statementDeleted(row):
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> [] }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response)
        print(row)
        return 88

# Example usage
print(statementDeleted(df_type_repairs.iloc[0]))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_type_repairs.iterrows(), total=len(df_type_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['S_deleted']):
        result = statementDeleted(row)
        if result == 88:
            print("index value:")
            print(index)
        else:
            df_type_repairs.at[index, 'S_deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 1000000 == 0 and index != 0:
        df_type_repairs.to_csv("checkpoint_type.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_type_repairs.to_csv("checkpoint_type.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_type_repairs.iloc[4919]

In [ ]:
df_type_repairs = df_type_repairs.drop(4919)

In [ ]:
len(df_type_repairs[(df_type_repairs['S_deleted'] == True)] )

In [ ]:
df_type_repairs[
    (df_type_repairs['C_deleted'] == False)
    & (df_type_repairs['C_deprecated'] == False)
    & (df_type_repairs['CQ_added_exception'] == False)
    & (df_type_repairs['S_deleted'] == False)
]

- testing for context statement addition (type triple) and t-box class hierarchy fixes:

In [ ]:
df_type_repairs['C_added_hierarchy'] = None
df_type_repairs['Sc_added_type'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getType(row, endpoint):

    if row['constraint_type'] == 'instance_of':
        string_path_type = '<'+ row['subject']+'> wdt:P31 ?type.'
    elif row['constraint_type'] == 'subclass_of':
        string_path_type = '<'+ row['subject']+'> wdt:P279 ?type.'
    elif row['constraint_type'] == 'instance_or_subclass':
        string_path_type = '<'+ row['subject']+'> wdt:P31|wdt:P279 ?type.'
    else:
        return None

    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        SELECT 
            ?type
        {{
          {string_path_type}
        }}
            """
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
       # Parse the XML
        root = ET.fromstring(response.text)

        # Define the namespace
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Create an empty set to store the type URIs
        type_set = set()

        # Find all URI elements corresponding to the variable 'type'
        for uri in root.findall(".//ns:binding[@name='type']/ns:uri", namespace) or []:
            type_set.add(uri.text)
            
        return type_set
    else:
        # If there's an error in the request, return None
        print("Error:", response)
        print(row)
        return set()
    
def typeStatementAdded(row):
    setType19 = getType(row, "ENTER_qEndpoint_WD_2019")
    setType23 = getType(row, "ENTER_qEndpoint_WD_2023")
    if setType19 is None or setType23 is None:
        return None
    if len(setType23) == 0:
        return False
    if setType19 == setType23:
        return False
    if len(setType23) > len(setType19):
        return True
    if setType19 != setType23:
        return True
    
    
def hierarchyAdded(row):
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"
    
    # get the types in 2019
    set_type = getType(row, "ENTER_qEndpoint_WD_2019")
    if len(set_type) == 0:
        return False

    for class_type in set_type:
        
        # regardless of the constraint type, they all are now checked with subclass path because the first step was calculated
        # in the getType function
        string_path_type = '<'+ class_type +'> wdt:P279* ?class.'
    
        # SPARQL query
        query = f"""
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            ASK
            {{
              <{row['property']}> p:P2302 ?statement.
              ?statement ps:P2302 wd:Q21503250.
              ?statement pq:P2308 ?class.
              {string_path_type}
            }}
        """
        #print(query)
        # URL encode the query
        encoded_query = requests.utils.quote(query)

        # Build the complete URL
        url = f"{endpoint}?query={encoded_query}"

        # Send HTTP GET request
        headers = {"Accept": "application/xhtml+xml,application/xml;"}

        response = requests.get(url,headers=headers)
        #print(response.text)
        # Check if the request was successful and parse the response
        if response.ok:
            # Parse the XML response
            #print(response.text)
            root = ET.fromstring(response.text)
            boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
            if boolean_element is not None:
                #print(boolean_element.text)
                if boolean_element.text.lower() == 'true':
                    return True
            else:
                print("Error: 'boolean' element not found in XML response")
        else:
            # If there's an error in the request, return None
            print("Error:", response)
            print(row)
            
    return False

# Example usage
#print(hierarchyAdded(df_type_repairs.iloc[2]))
#print(getType(df_type_repairs.iloc[2], "ENTER_qEndpoint_WD_2019"))
print(typeStatementAdded(df_type_repairs.iloc[3]))
print(hierarchyAdded(df_type_repairs.iloc[3]))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_type_repairs.iterrows(), total=len(df_type_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['C_added_hierarchy']):
        result = hierarchyAdded(row)
        df_type_repairs.at[index, 'C_added_hierarchy'] = result
        
    if pd.isna(row['Sc_added_type']):
        result = typeStatementAdded(row)
        df_type_repairs.at[index, 'Sc_added_type'] = result

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0 and index != 0:
        df_type_repairs.to_csv("checkpoint_type.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_type_repairs.to_csv("checkpoint_type.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_type_repairs['C_added_hierarchy'].value_counts()

In [ ]:
df_type_repairs[
    (df_type_repairs['C_deleted'] == False)
    & (df_type_repairs['C_deprecated'] == False)
    & (df_type_repairs['CQ_added_exception'] == False)
    & (df_type_repairs['S_deleted'] == False)
    & (df_type_repairs['C_added_hierarchy'] == False)
]

In [ ]:
df_type_repairs['C_added_hierarchy'].value_counts()

In [ ]:
df_type_repairs['Sc_added_type'].value_counts()

- testing changes in the expected relation:

In [ ]:
df_type_repairs['CQ_replace'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getRelation(row, endpoint):
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    
    # SPARQL query
    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        SELECT ?relation
        {{
          <{row['property']}> p:P2302 ?statement.
          ?statement ps:P2302 wd:Q21503250. ## subject-type constraint
          ?statement pq:P2309 ?relation.
          # no deprecated, no exceptions
          FILTER NOT EXISTS {{ ?statement pq:P2241 [] }}
          FILTER NOT EXISTS {{ ?statement wikibase:rank wikibase:DeprecatedRank }}
        }}
            """
    #print(query)
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element =  root.find(".//ns:binding[@name='relation']/ns:uri", namespace)
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response)
        print(row)
        return 88
    
def hasRelationChanged(row):
    relation_2019 = getRelation(row, 'ENTER_qEndpoint_WD_2019')
    if relation_2019 is None:
        return None
    relation_2023 = getRelation(row, 'ENTER_qEndpoint_WD_2023')
    if relation_2023 is None:
        return None
    if relation_2019 != relation_2023:
        return True
    return False

# Example usage
print(getRelation(df_type_repairs.iloc[4], 'ENTER_qEndpoint_WD_2019'))
print(getRelation(df_type_repairs.iloc[4], 'ENTER_qEndpoint_WD_2023'))
print(hasRelationChanged(df_type_repairs.iloc[4]))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_type_repairs.iterrows(), total=len(df_type_repairs)):
    
    if bool(row['C_deleted']) is True:
        df_type_repairs.at[index, 'CQ_replace'] = False
    elif pd.isna(row['CQ_replace']):
        result = hasRelationChanged(row)
        if result == 88:
            print("index value:")
            print(index)
        else:
            df_type_repairs.at[index, 'CQ_replace'] = result

    # Save a checkpoint every 10,000 rows
    if index % 300000 == 0 and index != 0:
        df_type_repairs.to_csv("checkpoint_type_2.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_type_repairs.to_csv("checkpoint_type_2.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_type_repairs['CQ_replace'].value_counts()

- revalidating remaning rows:

In [ ]:
df_unknown = df_type_repairs[
    (df_type_repairs['C_deleted'] == False)
    & (df_type_repairs['C_deprecated'] == False)
    & (df_type_repairs['CQ_added_exception'] == False)
    & (df_type_repairs['S_deleted'] == False)
    & (df_type_repairs['C_added_hierarchy'] == False)
    & (df_type_repairs['Sc_added_type'] == False)
    & (df_type_repairs['CQ_replace'] == False)
]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isStillViolation(row, endpoint):
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    
    if row['constraint_type'] == 'instance_of':
        string_path_type = '<'+ row['subject']+'> wdt:P31/wdt:P279* ?allowed_type.'
    elif row['constraint_type'] == 'subclass_of':
        string_path_type = '<'+ row['subject']+'> wdt:P279+ ?allowed_type.'
    elif row['constraint_type'] == 'instance_or_subclass':
        string_path_type = '<'+ row['subject']+'> wdt:P31/wdt:P279*|wdt:P279+ ?allowed_type.'
    else:
        return None
    
    c_type = ''
    if bool(row['CQ_replace']) == False:
        if row['constraint_type'] == 'instance_of':
            c_type = '?statement pq:P2309 wd:Q21503252.'
        elif row['constraint_type'] == 'subclass_of':
            c_type = '?statement pq:P2309 wd:Q21514624.'
        else:
            c_type = '?statement pq:P2309 wd:Q30208840.'
    
    # SPARQL query
    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        ASK
        {{
          ?subject <{wdt_pid}> [].

          <{row['property']}> p:P2302 ?statement.
          ?statement ps:P2302 wd:Q21503250. ## type constraint
          {c_type}

          # no deprecated, no exceptions
          FILTER NOT EXISTS {{?statement pq:P2241 []}}
          FILTER NOT EXISTS {{?statement wikibase:rank wikibase:DeprecatedRank}}
          FILTER NOT EXISTS {{?statement pq:P2303 ?subject}}
          FILTER (?subject = <{row['subject']}>)
          FILTER NOT EXISTS {{
            ?statement pq:P2308 ?allowed_type.
            {string_path_type}
          }}
        }}
            """
    #print(query)
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response)
        print(row)
        return 88

# Example usage

In [ ]:
isStillViolation(df_unknown.iloc[2], "ENTER_qEndpoint_WD_2023")

In [ ]:
df_unknown['still_violation'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_unknown.iterrows(), total=len(df_unknown)):
    
    # Check for unprocessed rows
    if pd.isna(row['still_violation']) or row['still_violation'] == None:
        result = isStillViolation(row, "ENTER_qEndpoint_WD_2023")
        if result == 88:
            print("index value:")
            print(index)
        else:
            df_unknown.at[index, 'still_violation'] = result

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0 and index != 0:
        df_unknown.to_csv("checkpoint_unknown.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_unknown.to_csv("checkpoint_unknown.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_unknown['still_violation'].value_counts()

In [ ]:
# Step 1: Get indices of rows in df_unknown where still_violation is True
violation_indices = df_unknown[df_unknown['still_violation'] == True].index

# Step 2: Remove rows with these indices from df_type_repairs
df_type_repairs = df_type_repairs.drop(violation_indices)

- check for new allowed classes in 2023 (CQ_added_class)

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getTboxClassList(row, endpoint): # = "ENTER_qEndpoint_WD_2019"

    pid = row['property'].replace("http://www.wikidata.org/entity/", "")
    # SPARQL query
    query = f"""
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    PREFIX wikibase: <http://wikiba.se/ontology#>
    PREFIX p: <http://www.wikidata.org/prop/>
    PREFIX ps: <http://www.wikidata.org/prop/statement/>
    PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX bd: <http://www.bigdata.com/rdf#>

    SELECT ?class
    {{
      wd:{pid} p:P2302 ?CQ. 
      ?CQ ps:P2302 wd:Q21503250. 
      ?CQ pq:P2308 ?class.
      FILTER NOT EXISTS {{?CQ wikibase:rank wikibase:DeprecatedRank}}
    }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)

    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"

    # Send HTTP GET request
    headers = {"Accept": "application/sparql-results+xml"}

    response = requests.get(url, headers=headers)
    
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find all 'uri' elements and extract their text
        uris = [uri_element.text for uri_element in root.findall('.//ns:uri', namespace)]

        return uris  # <- Just return everything found
    else:
        print("Error:", response.text)
        return None
    

def hasType(row, class_to_test, endpoint = "ENTER_qEndpoint_WD_2019"):
     
    if row['constraint_type'] == 'instance_of':
        string_path_type = '<'+ row['subject']+'> wdt:P31/wdt:P279* <'+class_to_test+'>'
    elif row['constraint_type'] == 'subclass_of':
        string_path_type = '<'+ row['subject']+'> wdt:P279+ <'+class_to_test+'>'
    elif row['constraint_type'] == 'instance_or_subclass':
        string_path_type = '<'+ row['subject']+'> wdt:P31/wdt:P279*|wdt:P279+ <'+class_to_test+'>'
    else:
        return None
    # SPARQL query
    query = f"""ASK {{ {string_path_type} }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response)
        print(row)
        return 88

    
def added_already_existent_class_to_constraint(row):
    class_list_2023 = getTboxClassList(row, "ENTER_qEndpoint_WD_2023")
    if class_list_2023 is None or len(class_list_2023) == 0:
        return False
    class_list_2019 = getTboxClassList(row, "ENTER_qEndpoint_WD_2019")
    if class_list_2019 is None or len(class_list_2019) == 0:
        return False
    new_types = [item for item in class_list_2023 if item not in class_list_2019]
    for nt in new_types:
        if hasType(row, nt):
            return True
    return False

#class_list =  getTboxClassList(df_type_repairs.iloc[0], "ENTER_qEndpoint_WD_2019")
#print(class_list)
#print(getTboxClassList(df_type_repairs.iloc[0], "ENTER_qEndpoint_WD_2023"))
#print(hasType(df_type_repairs.iloc[0], class_list[0], ))
added_already_existent_class_to_constraint(df_type_repairs.iloc[2717719])

In [ ]:
from tqdm import tqdm
import os

df_type_repairs['CQ_added_class'] = None

for index, row in tqdm(df_type_repairs.iterrows(), total=len(df_type_repairs)):
    
    if row['CQ_added_class'] is None:
        if row['C_deleted'] is True:
            df_type_repairs.at[index, 'CQ_added_class'] = False
        else:
            df_type_repairs.at[index, 'CQ_added_class'] = added_already_existent_class_to_constraint(row)

        # Save a checkpoint every 10,000 rows
        if index % 1000000 == 0 and index != 0:
            df_type_repairs.to_csv("checkpoint_type.csv", index=False)
            print(f"Checkpoint saved at row {index}.")

In [ ]:
df_type_repairs.to_csv("type_repairs.csv", index=False)